In [ ]:
!pip -q install lightkurve astropy pandas requests scikit-learn

import os
import io
import zipfile
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightkurve as lk
import torch

from astropy.time import Time
from sklearn.model_selection import GroupShuffleSplit
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Устройство:", DEVICE)
print("PyTorch:", torch.__version__)

In [ ]:

# =========================================
# ЯЧЕЙКА №1 — ЗАГРУЗКА И ПОДГОТОВКА NASA CSV
# =========================================

import pandas as pd
import numpy as np

CSV_PATH = "YOUR_TRAINING_DATASETS"

df = pd.read_csv(CSV_PATH, comment="#")

# Проверяем разделитель
if len(df.columns) == 1 and "," in df.columns[0]:
    df = pd.read_csv(CSV_PATH, sep=",", comment="#")

df.columns = df.columns.str.strip().str.lower()

required = [
    "kepid",
    "koi_period",
    "koi_time0bk",
    "koi_duration",
    "koi_disposition",
    "kepoi_name"
]

missing = [col for col in required if col not in df.columns]

if missing:
    raise ValueError(f"Не найдены столбцы: {missing}")

# Приводим числовые данные
numeric_columns = [
    "kepid",
    "koi_period",
    "koi_time0bk",
    "koi_duration"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["koi_disposition"] = (
    df["koi_disposition"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Оставляем только два нужных класса
df = df[
    df["koi_disposition"].isin([
        "CONFIRMED",
        "FALSE POSITIVE"
    ])
].copy()

# Убираем строки с отсутствующими параметрами
df = df.dropna(subset=required).copy()

# Проверяем физически допустимые значения
df = df[
    (df["kepid"] > 0) &
    (df["koi_period"] > 0) &
    (df["koi_duration"] > 0)
].copy()

df["kepid"] = df["kepid"].astype(int)

# Выбираем одинаковое количество объектов каждого класса
N_PER_CLASS = 400

confirmed = df[df["koi_disposition"] == "CONFIRMED"]
false_positive = df[df["koi_disposition"] == "FALSE POSITIVE"]

n = min(
    N_PER_CLASS,
    len(confirmed),
    len(false_positive)
)

if n == 0:
    raise ValueError("В одном из классов нет подходящих объектов.")

df = pd.concat([
    confirmed.sample(n=n, random_state=42),
    false_positive.sample(n=n, random_state=42)
])

# Перемешиваем данные
df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Создаём числовые метки
df["target"] = df["koi_disposition"].map({
    "CONFIRMED": 0,
    "FALSE POSITIVE": 1
}).astype(int)

# Продолжительность транзита в днях
df["duration_days"] = df["koi_duration"] / 24.0

print("Количество объектов:", len(df))
print("Количество уникальных звёзд:", df["kepid"].nunique())

print("\nРаспределение классов:")
print(df["koi_disposition"].value_counts())

print("\nПервые строки:")
display(df[[
    "kepid",
    "kepoi_name",
    "koi_period",
    "koi_time0bk",
    "duration_days",
    "koi_disposition",
    "target"
]].head())


Количество объектов: 800
Количество уникальных звёзд: 775

Распределение классов:
koi_disposition
FALSE POSITIVE    400
CONFIRMED         400
Name: count, dtype: int64

Первые строки:


,kepid,kepoi_name,koi_period,koi_time0bk,duration_days,koi_disposition,target
0,10753734,K07371.01,19.406694,149.807826,0.219902,FALSE POSITIVE,1
1,2021440,K01987.01,7.811540,136.331010,0.249333,FALSE POSITIVE,1
2,9480310,K01470.01,15.434010,363.294940,0.177383,CONFIRMED,0
3,3532510,K04985.01,106.779168,146.114560,0.079375,FALSE POSITIVE,1
4,9022166,K02175.01,26.847596,142.170230,0.419500,CONFIRMED,0


In [ ]:

# =========================================
# ЯЧЕЙКА №2 — ПОДКЛЮЧЕНИЕ GOOGLE DRIVE И КЭШ
# =========================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Основная папка проекта
PROJECT_DIR = Path(
    "YOUR_PROGECT_PATH"
)

# Кэш световых кривых
CACHE_DIR = PROJECT_DIR / "lightcurve_cache"

# Подготовленные сигналы
RESULTS_DIR = PROJECT_DIR / "prepared_data"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Файлы промежуточного сохранения
X_PATH = RESULTS_DIR / "X.npy"
Y_PATH = RESULTS_DIR / "y.npy"
GROUPS_PATH = RESULTS_DIR / "groups.npy"

INFO_PATH = RESULTS_DIR / "prepared_info.csv"
ERRORS_PATH = RESULTS_DIR / "errors.csv"

print("Папки созданы.")
print("Кэш:", CACHE_DIR)
print("Результаты:", RESULTS_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Папки созданы.
Кэш: /content/drive/MyDrive/PlanetX/lightcurve_cache
Результаты: /content/drive/MyDrive/PlanetX/prepared_data


In [ ]:
# =========================================
# ЯЧЕЙКА №3 — ЗАГРУЗКА LIGHTKURVE (С ТАЙМАУТАМИ И RETRY)
# =========================================

import time as time_module
import random
from pathlib import Path

import numpy as np
import pandas as pd
import lightkurve as lk

from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
from astropy.utils.data import conf as astropy_conf
from astroquery.mast import Observations

# -----------------------------------------
# НАСТРОЙКИ
# -----------------------------------------

INPUT_LENGTH = 2000
MAX_STARS = None          # например 150, чтобы быстро протестировать
SAVE_EVERY = 50           # сохранять каждые N звёзд
MAX_WORKERS = 4           # одновременных загрузок
MAX_QUARTERS = None       # например 8: меньше файлов, но меньше транзитов
STALL_TIMEOUT = 180       # секунд без прогресса -> остановка с сохранением

# Таймауты сети, чтобы запросы не висели вечно
astropy_conf.remote_timeout = 30
Observations.TIMEOUT = 30

# -----------------------------------------
# ПУТИ (если не заданы в предыдущих ячейках)
# -----------------------------------------

if "RESULTS_DIR" not in globals():
    RESULTS_DIR = Path("YOUR_PROGECT/RESULTS")
RESULTS_DIR = Path(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if "CACHE_DIR" not in globals():
    CACHE_DIR = RESULTS_DIR / "cache"
CACHE_DIR = Path(CACHE_DIR)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

if "X_PATH" not in globals():
    X_PATH = RESULTS_DIR / "X.npy"
if "Y_PATH" not in globals():
    Y_PATH = RESULTS_DIR / "y.npy"
if "GROUPS_PATH" not in globals():
    GROUPS_PATH = RESULTS_DIR / "groups.npy"
if "INFO_PATH" not in globals():
    INFO_PATH = RESULTS_DIR / "info.csv"
if "ERRORS_PATH" not in globals():
    ERRORS_PATH = RESULTS_DIR / "errors.csv"

X_PATH, Y_PATH, GROUPS_PATH = Path(X_PATH), Path(Y_PATH), Path(GROUPS_PATH)
INFO_PATH, ERRORS_PATH = Path(INFO_PATH), Path(ERRORS_PATH)

# Кэш FITS-файлов lightkurve (лучше держать на Google Drive)
LK_CACHE = CACHE_DIR / "fits"
LK_CACHE.mkdir(parents=True, exist_ok=True)


# =========================================
# ПРОВЕРКА ТАБЛИЦЫ
# =========================================

required_columns = [
    "kepid", "kepoi_name", "koi_period",
    "koi_time0bk", "duration_days", "target"
]

missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"В таблице отсутствуют столбцы: {missing_columns}")

work_df = df.dropna(subset=required_columns).copy()
work_df["kepid"] = work_df["kepid"].astype(int)

work_df = work_df[
    (work_df["koi_period"] > 0) &
    (work_df["duration_days"] > 0) &
    (work_df["target"].isin([0, 1]))
].copy()


# =========================================
# ПОДГОТОВКА ОДНОГО СИГНАЛА
# =========================================

def prepare_signal(lc, period, epoch, duration_days):

    t = np.asarray(lc.time.value, dtype=float)
    flux = np.asarray(lc.flux.value, dtype=float)

    valid = np.isfinite(t) & np.isfinite(flux)
    t = t[valid]
    flux = flux[valid]

    if len(t) < 100:
        return None

    median_flux = np.nanmedian(flux)
    if not np.isfinite(median_flux) or median_flux == 0:
        return None

    flux = flux / median_flux - 1.0

    # Фаза относительно эпохи транзита
    phase_days = ((t - epoch + period / 2) % period) - period / 2
    phase = phase_days / duration_days

    # Окно ±5 длительностей транзита
    mask = np.abs(phase) <= 5
    phase = phase[mask]
    flux = flux[mask]

    if len(phase) < 20:
        return None

    order = np.argsort(phase)
    phase = phase[order]
    flux = flux[order]

    # Усредняем повторяющиеся фазы
    unique_phase, inverse = np.unique(phase, return_inverse=True)
    summed_flux = np.bincount(inverse, weights=flux)
    counts = np.bincount(inverse)
    mean_flux = summed_flux / counts

    if len(unique_phase) < 20:
        return None

    if unique_phase[0] > -4.5 or unique_phase[-1] < 4.5:
        return None

    target_phase = np.linspace(-5, 5, INPUT_LENGTH)
    interpolated_flux = np.interp(target_phase, unique_phase, mean_flux)

    std = np.std(interpolated_flux)
    if not np.isfinite(std) or std < 1e-10:
        return None

    interpolated_flux = (interpolated_flux - np.mean(interpolated_flux)) / std

    return interpolated_flux.astype(np.float32)


# =========================================
# RETRY И ЗАГРУЗКА КРИВОЙ С КЭШИРОВАНИЕМ
# =========================================

def with_retry(fn, tries=4):
    for i in range(tries):
        try:
            return fn()
        except Exception:
            if i == tries - 1:
                raise
            time_module.sleep(2 ** i + random.random())


def load_star_curve(kepid):

    cache_file = CACHE_DIR / f"kepid_{kepid}.npz"

    # Проверяем наличие кэша
    if cache_file.exists():
        try:
            with np.load(cache_file) as cached:
                t = cached["time"]
                f = cached["flux"]
            return lk.LightCurve(time=t, flux=f), "cache"
        except Exception:
            cache_file.unlink(missing_ok=True)   # битый кэш — качаем заново

    # Поиск данных Kepler
    search_result = with_retry(lambda: lk.search_lightcurve(
        f"KIC {kepid}", mission="Kepler", cadence="long"
    ))

    if len(search_result) == 0:
        raise ValueError("Кривая блеска не найдена")

    if MAX_QUARTERS:
        search_result = search_result[:MAX_QUARTERS]

    # Загрузка наблюдений
    collection = with_retry(lambda: search_result.download_all(
        download_dir=str(LK_CACHE)
    ))

    if collection is None or len(collection) == 0:
        raise ValueError("Не удалось скачать кривые блеска")

    lc = collection.stitch().remove_nans()

    if len(lc.time) < 100:
        raise ValueError("Недостаточно точек")

    t = np.asarray(lc.time.value, dtype=np.float64)
    f = np.asarray(lc.flux.value, dtype=np.float32)

    # Пишем во временный файл, потом переименовываем (защита от битого кэша)
    tmp_file = CACHE_DIR / f"kepid_{kepid}.tmp.npz"
    np.savez(tmp_file, time=t, flux=f)
    tmp_file.replace(cache_file)

    return lk.LightCurve(time=t, flux=f), "download"


# =========================================
# ВОССТАНОВЛЕНИЕ ПРЕДЫДУЩИХ РЕЗУЛЬТАТОВ
# =========================================

if X_PATH.exists() and Y_PATH.exists() and GROUPS_PATH.exists():

    X = list(np.load(X_PATH))
    y = list(np.load(Y_PATH))
    groups = list(np.load(GROUPS_PATH))

    prepared_info = (
        pd.read_csv(INFO_PATH).to_dict("records")
        if INFO_PATH.exists() else []
    )

    # Ошибки старого запуска не восстанавливаем: ошибочные объекты
    # будут обработаны заново
    errors = []

    print("Найдены ранее сохранённые результаты.")

else:
    X, y, groups = [], [], []
    prepared_info, errors = [], []


# =========================================
# ОПРЕДЕЛЕНИЕ УЖЕ ОБРАБОТАННЫХ KOI
# =========================================

completed_names = {str(item["kepoi_name"]) for item in prepared_info}

star_groups = list(work_df.groupby("kepid"))

if MAX_STARS is not None:
    star_groups = star_groups[:MAX_STARS]

remaining_groups = []

for kepid, star_rows in star_groups:
    remaining_rows = star_rows[
        ~star_rows["kepoi_name"].astype(str).isin(completed_names)
    ]
    if len(remaining_rows) > 0:
        remaining_groups.append((int(kepid), remaining_rows))

print("Всего уникальных звёзд:", len(star_groups))
print("Звёзд с необработанными объектами:", len(remaining_groups))
print("Уже успешно подготовлено:", len(prepared_info))


# =========================================
# СОХРАНЕНИЕ ПРОМЕЖУТОЧНЫХ РЕЗУЛЬТАТОВ
# =========================================

def save_progress():
    np.save(X_PATH, np.asarray(X, dtype=np.float32))
    np.save(Y_PATH, np.asarray(y, dtype=np.int64))
    np.save(GROUPS_PATH, np.asarray(groups, dtype=np.int64))
    pd.DataFrame(prepared_info).to_csv(INFO_PATH, index=False)
    pd.DataFrame(errors).to_csv(ERRORS_PATH, index=False)


# =========================================
# ОБРАБОТКА ОДНОЙ ЗВЕЗДЫ
# =========================================

def process_star(kepid, star_rows, lc):

    for row in star_rows.itertuples(index=False):

        name = str(row.kepoi_name)

        try:
            signal = prepare_signal(
                lc=lc,
                period=float(row.koi_period),
                epoch=float(row.koi_time0bk),
                duration_days=float(row.duration_days)
            )

            if signal is None:
                errors.append({
                    "kepid": kepid,
                    "kepoi_name": name,
                    "reason": "Не удалось подготовить участок транзита"
                })
                continue

            X.append(signal)
            y.append(int(row.target))
            groups.append(kepid)
            prepared_info.append({
                "kepid": kepid,
                "kepoi_name": name,
                "target": int(row.target)
            })

        except Exception as e:
            errors.append({
                "kepid": kepid,
                "kepoi_name": name,
                "reason": str(e)
            })


# =========================================
# ПАРАЛЛЕЛЬНАЯ ЗАГРУЗКА (СКОЛЬЗЯЩЕЕ ОКНО)
# =========================================

downloaded_count = 0
cache_count = 0
completed_stars = 0

progress = tqdm(
    total=len(remaining_groups),
    desc="Загрузка и обработка звёзд",
    unit="звезда"
)

star_iter = iter(remaining_groups)
pending = {}

executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)


def submit_next():
    try:
        kepid, rows = next(star_iter)
    except StopIteration:
        return
    pending[executor.submit(load_star_curve, kepid)] = (kepid, rows)


try:
    # Заполняем очередь
    for _ in range(MAX_WORKERS * 2):
        submit_next()

    while pending:

        done, _ = wait(
            pending,
            return_when=FIRST_COMPLETED,
            timeout=STALL_TIMEOUT
        )

        if not done:
            print(
                f"\nНет прогресса {STALL_TIMEOUT} сек — возможно, MAST недоступен. "
                "Останавливаюсь, прогресс сохранён. Запусти ячейку заново."
            )
            break

        for f in done:

            kepid, star_rows = pending.pop(f)

            try:
                lc, source = f.result()

                if source == "cache":
                    cache_count += 1
                else:
                    downloaded_count += 1

                process_star(kepid, star_rows, lc)
                del lc

            except Exception as e:
                for row in star_rows.itertuples(index=False):
                    errors.append({
                        "kepid": kepid,
                        "kepoi_name": str(row.kepoi_name),
                        "reason": str(e)
                    })

            completed_stars += 1
            progress.update(1)
            progress.set_postfix({
                "KIC": kepid,
                "кэш": cache_count,
                "скачано": downloaded_count,
                "подготовлено": len(X)
            })

            if completed_stars % SAVE_EVERY == 0:
                save_progress()

            submit_next()

except KeyboardInterrupt:
    print("\nОстановлено вручную. Сохраняю прогресс...")

finally:
    # Не ждём зависшие потоки
    executor.shutdown(wait=False, cancel_futures=True)
    progress.close()
    save_progress()


# =========================================
# ИТОГИ
# =========================================

X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.int64)
groups = np.asarray(groups, dtype=np.int64)

prepared_info = pd.DataFrame(prepared_info)
errors_df = pd.DataFrame(errors)

print("\nОбработка завершена!")
print("Размер X:", X.shape)
print("Размер y:", y.shape)
print("Количество уникальных звёзд:", len(np.unique(groups)))

if len(y) > 0:
    print("\nРаспределение подготовленных классов:")
    print(
        pd.Series(y).map({0: "CONFIRMED", 1: "FALSE POSITIVE"}).value_counts()
    )

print("\nСкачано новых звёзд:", downloaded_count)
print("Взято из кэша:", cache_count)
print("Ошибок:", len(errors_df))
print("\nВсе результаты сохранены в:", RESULTS_DIR)

Найдены ранее сохранённые результаты.
Всего уникальных звёзд: 775
Звёзд с необработанными объектами: 278
Уже успешно подготовлено: 513


Загрузка и обработка звёзд:   0%|          | 0/278 [00:00<?, ?звезда/s]

 [Done]


In [ ]:

# =========================================
# ЯЧЕЙКА №4 — ПРОВЕРКА ДАТАСЕТА
# =========================================

import numpy as np
import pandas as pd

X = np.load(X_PATH)
y = np.load(Y_PATH)
groups = np.load(GROUPS_PATH)

prepared_info = pd.read_csv(INFO_PATH)
errors_df = pd.read_csv(ERRORS_PATH)

print("Размер X:", X.shape)
print("Размер y:", y.shape)
print("Размер groups:", groups.shape)

print("\nУникальных звёзд:", len(np.unique(groups)))

print("\nРаспределение классов:")

print(
    pd.Series(y).map({
        0: "CONFIRMED",
        1: "FALSE POSITIVE"
    }).value_counts()
)

print("\nКоличество ошибок:", len(errors_df))

print("\nПервые подготовленные объекты:")

display(prepared_info.head())

print("\nПервые ошибки:")

display(errors_df.head())

# Проверяем форму сигналов
assert X.ndim == 2
assert X.shape[1] == 2000
assert len(X) == len(y) == len(groups)

assert np.isfinite(X).all()

print("\nПроверка успешно пройдена!")


NameError: name 'X_PATH' is not defined

In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import numpy as np

# Проверяем подготовленные данные
if len(X) == 0:
    raise ValueError("Нет подготовленных сигналов. Сначала проверь ячейку 3.")

if len(np.unique(y)) != 2:
    raise ValueError(
        "Для обучения нужны оба класса: CONFIRMED и FALSE POSITIVE."
    )

if len(np.unique(groups)) < 2:
    raise ValueError("Недостаточно разных звёзд для разделения данных.")

# Разделяем данные по звёздам.
# Подбираем разделение, в котором присутствуют оба класса
splitter = GroupShuffleSplit(
    n_splits=100,
    test_size=0.2,
    random_state=42
)

split_found = False

for train_idx, val_idx in splitter.split(X, y, groups):

    if (
        len(np.unique(y[train_idx])) == 2 and
        len(np.unique(y[val_idx])) == 2
    ):
        split_found = True
        break

if not split_found:
    raise ValueError(
        "Не удалось разделить данные так, чтобы оба класса "
        "присутствовали в обучении и проверке. "
        "Нужно больше звёзд каждого класса."
    )

X_train = X[train_idx]
X_val = X[val_idx]

y_train = y[train_idx]
y_val = y[val_idx]

print("Обучающих сигналов:", len(X_train))
print("Проверочных сигналов:", len(X_val))

print("Звёзд в обучении:", len(np.unique(groups[train_idx])))
print("Звёзд в проверке:", len(np.unique(groups[val_idx])))

# Переводим данные в PyTorch
X_train_tensor = torch.tensor(X_train).unsqueeze(1)
X_val_tensor = torch.tensor(X_val).unsqueeze(1)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

# Архитектура CNN
class PlanetX_CNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv1d(1, 16, kernel_size=7, padding=3),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


# Выбираем устройство
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Устройство:", device)

model = PlanetX_CNN().to(device)

# Учитываем возможный дисбаланс классов
class_counts = np.bincount(y_train, minlength=2)

if np.any(class_counts == 0):
    raise ValueError("В обучающей выборке отсутствует один из классов.")

class_weights = len(y_train) / (2.0 * class_counts)

weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(
    weight=weights_tensor
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Обучение
EPOCHS = 25

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0.0

    for batch_X, batch_y in train_loader:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    # Проверка на валидационной выборке
    model.eval()

    predictions = []
    actual = []

    with torch.no_grad():

        for batch_X, batch_y in val_loader:

            batch_X = batch_X.to(device)

            outputs = model(batch_X)
            predicted = torch.argmax(outputs, dim=1)

            predictions.extend(predicted.cpu().numpy())
            actual.extend(batch_y.numpy())

    val_accuracy = accuracy_score(actual, predictions)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {average_loss:.4f} | "
        f"Validation accuracy: {val_accuracy:.4f}"
    )


# Итоговая оценка
print("\nИтоговая точность:", accuracy_score(actual, predictions))

print("\nОтчёт классификации:")
print(
    classification_report(
        actual,
        predictions,
        labels=[0, 1],
        target_names=[
            "CONFIRMED",
            "FALSE POSITIVE"
        ],
        zero_division=0
    )
)

print("\nМатрица ошибок:")
print(
    confusion_matrix(
        actual,
        predictions,
        labels=[0, 1]
    )
)

# Сохраняем модель
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "input_length": INPUT_LENGTH,
        "class_mapping": {
            0: "CONFIRMED",
            1: "FALSE POSITIVE"
        }
    },
    "planetx_cnn.pth"
)

print("\nМодель сохранена: planetx_cnn.pth")
